In [2]:
# ============================================================
# FILE 14 — Omics Global Signature Exploration
# CellLineFinder | Person 2 | AstraZeneca MSc Dissertation
# ============================================================

import pandas as pd
import numpy as np

PATH = "./data/non gene expression/14_OmicsGlobalSignatures.csv"  # adjust filename if different

# ============================================================
# BLOCK 1 — File size, separator, shape, columns
# ============================================================

import os
size_bytes = os.path.getsize(PATH)
print(f"File size: {size_bytes / 1e6:.1f} MB")

# Try comma, then tab
for sep in [",", "\t", ";"]:
    try:
        df_peek = pd.read_csv(PATH, sep=sep, nrows=3)
        if df_peek.shape[1] > 2:
            print(f"Separator detected: {repr(sep)}")
            break
    except Exception:
        continue

df = pd.read_csv(PATH, sep=sep, low_memory=False)
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"\nColumns:\n{list(df.columns)}")
print(f"\nDtypes:\n{df.dtypes}")
print(f"\nFirst 3 rows:\n{df.head(3).to_string()}")


File size: 0.3 MB
Separator detected: ','
Shape: 3,021 rows x 12 columns

Columns:
['Unnamed: 0', 'SequencingID', 'ModelID', 'ModelConditionID', 'IsDefaultEntryForModel', 'IsDefaultEntryForMC', 'MSIScore', 'LoHFraction', 'WGD', 'CIN', 'Ploidy', 'Aneuploidy']

Dtypes:
Unnamed: 0                  int64
SequencingID               object
ModelID                    object
ModelConditionID           object
IsDefaultEntryForModel     object
IsDefaultEntryForMC        object
MSIScore                  float64
LoHFraction               float64
WGD                       float64
CIN                       float64
Ploidy                    float64
Aneuploidy                float64
dtype: object

First 3 rows:
   Unnamed: 0 SequencingID     ModelID ModelConditionID IsDefaultEntryForModel IsDefaultEntryForMC  MSIScore  LoHFraction  WGD       CIN    Ploidy  Aneuploidy
0           0   CDS-00Nrci  ACH-000839   MC-000839-krru                    Yes                 Yes      3.68     0.107443  1.0  0.502634

In [3]:

# ============================================================
# BLOCK 2 — Orientation: is this wide or long format?
# ============================================================

# Wide = cell lines as rows, signatures as columns (or vice versa)
# Long = one value per row with cell_line + signature + value columns

print(f"\nRow count: {df.shape[0]:,}")
print(f"Column count: {df.shape[1]:,}")

# Heuristic: if columns >> rows, likely wide with signatures as columns
# If ~3-5 columns, likely long format
if df.shape[1] <= 10:
    print("Format hint: likely LONG format (few columns)")
else:
    print("Format hint: likely WIDE format (many columns)")

# Check first column — is it an index/ID?
first_col = df.columns[0]
print(f"\nFirst column '{first_col}':")
print(f"  Unique: {df[first_col].nunique():,}")
print(f"  Sample: {df[first_col].dropna().head(10).tolist()}")


Row count: 3,021
Column count: 12
Format hint: likely WIDE format (many columns)

First column 'Unnamed: 0':
  Unique: 3,021
  Sample: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


In [4]:

# ============================================================
# BLOCK 3 — Nulls
# ============================================================

null_counts = df.isnull().sum()
null_pct = (null_counts / len(df) * 100).round(1)
null_df = pd.DataFrame({"null_count": null_counts, "null_%": null_pct})
print("\nNulls (non-zero only):")
print(null_df[null_df["null_count"] > 0].sort_values("null_%", ascending=False).to_string())
print(f"\nTotal null cells: {df.isnull().sum().sum():,} of {df.size:,} ({df.isnull().sum().sum()/df.size*100:.1f}%)")


Nulls (non-zero only):
             null_count  null_%
LoHFraction         439    14.5
WGD                 439    14.5
CIN                 439    14.5
Ploidy              439    14.5
Aneuploidy          439    14.5

Total null cells: 2,195 of 36,252 (6.1%)


In [5]:
# ============================================================
# BLOCK 4 — Identify cell line column
# ============================================================

cl_candidates = [c for c in df.columns if any(
    kw in c.lower() for kw in [
        "cell", "line", "name", "sample", "model", "id",
        "ach", "cvcl", "ccle", "depmap"
    ]
)]
print(f"Candidate cell line columns: {cl_candidates}")

for col in cl_candidates:
    print(f"\n[{col}]")
    print(f"  Unique: {df[col].nunique():,}")
    print(f"  Sample: {df[col].dropna().head(10).tolist()}")

Candidate cell line columns: ['Unnamed: 0', 'SequencingID', 'ModelID', 'ModelConditionID', 'IsDefaultEntryForModel', 'Ploidy', 'Aneuploidy']

[Unnamed: 0]
  Unique: 3,021
  Sample: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

[SequencingID]
  Unique: 3,021
  Sample: ['CDS-00Nrci', 'CDS-051xn7', 'CDS-099jzP', 'CDS-0A4mDu', 'CDS-0Eax8o', 'CDS-0Fwtih', 'CDS-0GB1gy', 'CDS-0PlxsQ', 'CDS-0RmKXB', 'CDS-0V2XTR']

[ModelID]
  Unique: 1,955
  Sample: ['ACH-000839', 'ACH-000041', 'ACH-002046', 'ACH-002048', 'ACH-000042', 'ACH-002433', 'ACH-001669', 'ACH-001300', 'ACH-001574', 'ACH-000332']

[ModelConditionID]
  Unique: 2,564
  Sample: ['MC-000839-krru', 'MC-000041-uPBf', 'MC-002046-oaX8', 'MC-002048-52d6', 'MC-000042-eOnX', 'MC-002433-iccX', 'MC-001669-PygV', 'MC-001300-8ioY', 'MC-001574-0hwo', 'MC-000332-21ni']

[IsDefaultEntryForModel]
  Unique: 2
  Sample: ['Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'No', 'Yes', 'Yes', 'Yes', 'Yes']

[Ploidy]
  Unique: 2,582
  Sample: [3.15829106158172, 3.23608901214141, 3.3267145

In [6]:
# ============================================================
# BLOCK 5 — Identify signature / feature columns
# ============================================================

# What are the non-ID columns?
# If wide: columns are likely signature names
# If long: there will be a signature name column

sig_candidates = [c for c in df.columns if any(
    kw in c.lower() for kw in [
        "sig", "pathway", "score", "activity", "hallmark",
        "gsea", "ssgsea", "enrichment", "feature"
    ]
)]
print(f"\nSignature/feature column candidates: {sig_candidates}")

# If wide format — summarise the value columns
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"\nNumeric columns: {len(numeric_cols)}")
if len(numeric_cols) > 0:
    print(f"  Sample names: {numeric_cols[:10]}")
    print(f"\nNumeric value range across all columns:")
    print(f"  Min: {df[numeric_cols].min().min():.4f}")
    print(f"  Max: {df[numeric_cols].max().max():.4f}")
    print(f"  Mean: {df[numeric_cols].mean().mean():.4f}")
    print(f"  % zeros: {(df[numeric_cols] == 0).sum().sum() / df[numeric_cols].size * 100:.1f}%")


Signature/feature column candidates: ['MSIScore']

Numeric columns: 7
  Sample names: ['Unnamed: 0', 'MSIScore', 'LoHFraction', 'WGD', 'CIN', 'Ploidy', 'Aneuploidy']

Numeric value range across all columns:
  Min: 0.0000
  Max: 3020.0000
  Mean: 219.6385
  % zeros: 4.9%


In [8]:
# ============================================================
# BLOCK 6 — If wide format: rows are cell lines, cols are signatures
#           Characterise both dimensions
# ============================================================

if df.shape[1] > 20:
    print("\n--- WIDE FORMAT ANALYSIS ---")

    # Row index — what identifies each row?
    print(f"\nRow identifier (first column) sample:")
    print(df.iloc[:, 0].head(20).tolist())

    # Column names — what are the signatures?
    print(f"\nAll column names:")
    print(df.columns.tolist())

    # Distribution of values in a sample of columns
    sample_cols = numeric_cols[:5]
    print(f"\nDescribe sample columns {sample_cols}:")
    print(df[sample_cols].describe().to_string())


In [9]:
# ============================================================
# BLOCK 7 — If long format: characterise the signature name column
# ============================================================

if df.shape[1] <= 10:
    print("\n--- LONG FORMAT ANALYSIS ---")
    for col in df.columns:
        n = df[col].nunique()
        print(f"\n[{col}] — {n} unique values")
        if n <= 200:
            print(df[col].value_counts(dropna=False).head(30).to_string())
        else:
            print(f"  Sample: {df[col].dropna().head(10).tolist()}")


In [10]:

# ============================================================
# BLOCK 8 — Cell line name format: CVCL, ACH-, symbol, or raw name?
# ============================================================

# Try to detect the format of cell line identifiers
for col in cl_candidates:
    vals = df[col].dropna().astype(str)
    cvcl = vals.str.startswith("CVCL_").sum()
    ach  = vals.str.startswith("ACH-").sum()
    pr   = vals.str.startswith("PR-").sum()
    ensg = vals.str.startswith("ENSG").sum()
    print(f"\n[{col}] ID format breakdown:")
    print(f"  CVCL_: {cvcl}")
    print(f"  ACH-:  {ach}")
    print(f"  PR-:   {pr}")
    print(f"  ENSG:  {ensg}")
    print(f"  Other: {len(vals) - cvcl - ach - pr - ensg}")


[Unnamed: 0] ID format breakdown:
  CVCL_: 0
  ACH-:  0
  PR-:   0
  ENSG:  0
  Other: 3021

[SequencingID] ID format breakdown:
  CVCL_: 0
  ACH-:  0
  PR-:   0
  ENSG:  0
  Other: 3021

[ModelID] ID format breakdown:
  CVCL_: 0
  ACH-:  3021
  PR-:   0
  ENSG:  0
  Other: 0

[ModelConditionID] ID format breakdown:
  CVCL_: 0
  ACH-:  0
  PR-:   0
  ENSG:  0
  Other: 3021

[IsDefaultEntryForModel] ID format breakdown:
  CVCL_: 0
  ACH-:  0
  PR-:   0
  ENSG:  0
  Other: 3021

[Ploidy] ID format breakdown:
  CVCL_: 0
  ACH-:  0
  PR-:   0
  ENSG:  0
  Other: 2582

[Aneuploidy] ID format breakdown:
  CVCL_: 0
  ACH-:  0
  PR-:   0
  ENSG:  0
  Other: 2582


In [11]:
# ============================================================
# BLOCK 9 — Sanity check: known cell lines present?
# ============================================================

if cl_candidates:
    cl_col = cl_candidates[0]
    known = ["A-431", "A431", "SK-BR-3", "SKBR3", "HeLa", "HELA",
             "MCF7", "MCF-7", "HEK293", "SW480", "NCI-H1299", "H1299"]
    for name in known:
        hits = df[df[cl_col].astype(str).str.upper() == name.upper()]
        if len(hits) > 0:
            print(f"  FOUND: {name} ({len(hits)} row(s))")


In [12]:
# ============================================================
# BLOCK 10 — Signature type detection
#            What kind of signatures are these?
#            (Mutational, pathway activity, GSEA hallmarks, etc.)
# ============================================================

# If column names are the signatures (wide format)
if df.shape[1] > 20:
    cols_lower = [c.lower() for c in df.columns]
    checks = {
        "COSMIC mutational sig (SBS)": sum("sbs" in c for c in cols_lower),
        "COSMIC mutational sig (ID/DBS)": sum(c.startswith("id") or c.startswith("dbs") for c in cols_lower),
        "Hallmark pathways": sum("hallmark" in c for c in cols_lower),
        "KEGG pathways": sum("kegg" in c for c in cols_lower),
        "Reactome": sum("reactome" in c for c in cols_lower),
        "ssGSEA scores": sum("gsea" in c for c in cols_lower),
        "CNV/CNA": sum("cnv" in c or "cna" in c or "copy" in c for c in cols_lower),
        "Immune": sum("immune" in c or "tcell" in c or "nk" in c for c in cols_lower),
    }
    print("\nSignature type detection (keyword hits in column names):")
    for k, v in checks.items():
        print(f"  {k}: {v}")

# If long format — check a signature name column
if df.shape[1] <= 10 and sig_candidates:
    sig_col = sig_candidates[0]
    vals_lower = df[sig_col].dropna().str.lower()
    checks = {
        "COSMIC SBS": vals_lower.str.contains("sbs").sum(),
        "Hallmark": vals_lower.str.contains("hallmark").sum(),
        "KEGG": vals_lower.str.contains("kegg").sum(),
        "Reactome": vals_lower.str.contains("reactome").sum(),
        "Immune": vals_lower.str.contains("immune").sum(),
    }
    print("\nSignature type detection (keyword hits in signature name column):")
    for k, v in checks.items():
        print(f"  {k}: {v}")


In [13]:


# ============================================================
# BLOCK 11 — Value distribution: are scores continuous, binary, integer?
# ============================================================

if len(numeric_cols) > 0:
    all_vals = df[numeric_cols].values.flatten()
    all_vals = all_vals[~np.isnan(all_vals)]

    print(f"\nValue distribution across all numeric entries:")
    print(f"  Min:    {all_vals.min():.4f}")
    print(f"  Max:    {all_vals.max():.4f}")
    print(f"  Mean:   {all_vals.mean():.4f}")
    print(f"  Median: {np.median(all_vals):.4f}")
    print(f"  Std:    {all_vals.std():.4f}")

    unique_vals = np.unique(all_vals)
    print(f"  Unique numeric values: {len(unique_vals):,}")
    if len(unique_vals) <= 20:
        print(f"  All values: {unique_vals.tolist()}")
    else:
        print(f"  Sample values: {unique_vals[:10].tolist()} ...")

    # Binary check
    if set(unique_vals).issubset({0, 1, 0.0, 1.0}):
        print("  -> Looks BINARY (0/1)")
    elif all_vals.min() >= 0 and all_vals.max() <= 1:
        print("  -> Looks like PROBABILITIES or normalised scores (0-1 range)")
    elif all_vals.min() < 0:
        print("  -> Contains negative values — likely z-scores or enrichment scores")
    else:
        print("  -> Positive-only, potentially raw counts or activity scores")





Value distribution across all numeric entries:
  Min:    0.0000
  Max:    3020.0000
  Mean:   244.5886
  Median: 1.9885
  Std:    651.8939
  Unique numeric values: 11,465
  Sample values: [0.0, 2.95036117726701e-07, 2.95409994835289e-07, 3.46078358857662e-07, 3.46506785700807e-07, 1.1352085737987e-06, 1.13602612167031e-06, 3.38475050124892e-06, 7.90052234791906e-06, 7.9014921920754e-06] ...
  -> Positive-only, potentially raw counts or activity scores


In [14]:
# ============================================================
# BLOCK 12 — Overlap with DepMap cell lines (ACH- IDs from File 9)
#            and HPA cell lines (from File 1/11)
#            Run only if you have those files available
# ============================================================

print("""
--- OVERLAP CHECK (run if other files available) ---

# DepMap overlap (File 9 sample_info)
sample_info = pd.read_csv("9_DepMap_sample_info.csv")
depmap_ids = set(sample_info["DepMap_ID"].dropna())
file14_ids = set(df[cl_col].dropna().astype(str))
print(f"Overlap with DepMap ACH- IDs: {len(depmap_ids & file14_ids)}")

# HPA overlap (File 11)
hpa = pd.read_csv("11_HPA_description.csv", sep="\\t")
hpa_norm = set(hpa["Cell line"].str.replace(r'[\\s\\-]', '', regex=True).str.upper())
file14_norm = set(df[cl_col].astype(str).str.replace(r'[\\s\\-]', '', regex=True).str.upper())
print(f"Overlap with HPA cell lines (normalised): {len(hpa_norm & file14_norm)}")
""")


--- OVERLAP CHECK (run if other files available) ---

# DepMap overlap (File 9 sample_info)
sample_info = pd.read_csv("9_DepMap_sample_info.csv")
depmap_ids = set(sample_info["DepMap_ID"].dropna())
file14_ids = set(df[cl_col].dropna().astype(str))
print(f"Overlap with DepMap ACH- IDs: {len(depmap_ids & file14_ids)}")

# HPA overlap (File 11)
hpa = pd.read_csv("11_HPA_description.csv", sep="\t")
hpa_norm = set(hpa["Cell line"].str.replace(r'[\s\-]', '', regex=True).str.upper())
file14_norm = set(df[cl_col].astype(str).str.replace(r'[\s\-]', '', regex=True).str.upper())
print(f"Overlap with HPA cell lines (normalised): {len(hpa_norm & file14_norm)}")



In [15]:
default = df[df["IsDefaultEntryForModel"] == "Yes"]
print(f"Rows after IsDefaultEntryForModel filter: {len(default)}")
print(f"Unique ACH- IDs after filter: {default['ModelID'].nunique()}")
print(f"Nulls in LoHFraction after filter: {default['LoHFraction'].isnull().sum()}")

Rows after IsDefaultEntryForModel filter: 1955
Unique ACH- IDs after filter: 1955
Nulls in LoHFraction after filter: 333


In [16]:
print(df["WGD"].value_counts(dropna=False).sort_index())
print(f"\nMSIScore distribution:")
print(df["MSIScore"].describe())
# MSI threshold: typically >3.5 is MSI-high
print(f"MSI-high (>3.5): {(df['MSIScore'] > 3.5).sum()}")

WGD
0.0     856
1.0    1726
NaN     439
Name: count, dtype: int64

MSIScore distribution:
count    3021.000000
mean        6.390755
std        16.701642
min         0.300000
25%         1.240000
50%         2.080000
75%         3.220000
max        93.220000
Name: MSIScore, dtype: float64
MSI-high (>3.5): 652


In [17]:
default = df[df["IsDefaultEntryForModel"] == "Yes"]

# Null pattern
null_rows = default[default["LoHFraction"].isnull()]
print(f"Null LoHFraction rows: {len(null_rows)}")
print(null_rows["MSIScore"].describe())

# Post-filter MSI-high
print(f"\nMSI-high after default filter: {(default['MSIScore'] > 3.5).sum()}")

# Aneuploidy range on default rows
print(f"\nAneuploidy value counts (default rows):")
print(default["Aneuploidy"].value_counts().sort_index().to_string())

Null LoHFraction rows: 333
count    333.000000
mean      11.368288
std       24.314163
min        0.300000
25%        1.810000
50%        2.470000
75%        3.690000
max       91.580000
Name: MSIScore, dtype: float64

MSI-high after default filter: 386

Aneuploidy value counts (default rows):
Aneuploidy
0.0     135
1.0      37
2.0      51
3.0      37
4.0      49
5.0      28
6.0      48
7.0      31
8.0      31
9.0      42
10.0     41
11.0     28
12.0     24
13.0     19
14.0     32
15.0     40
16.0     24
17.0     42
18.0     59
19.0     67
20.0     67
21.0     61
22.0     74
23.0     81
24.0     80
25.0     68
26.0     65
27.0     51
28.0     42
29.0     41
30.0     25
31.0     21
32.0     21
33.0     17
34.0     13
35.0      9
36.0      5
37.0      7
38.0      6
39.0      3
